# Détecter l'idée de "République"

In [1]:
import pandas as pd
import re
import csv

df = pd.read_csv(
    "../data/interim/df_repu_proportion.csv", low_memory=False, dtype={"ID_orateur": str}
)
df.shape

(683660, 54)

In [ ]:
def analyse_regex_famille(texte: str, motifs: dict, nom_famille: str = None):
    """
    Analyse un texte à partir d'une famille de motifs regex.
    
    Args:
        texte (str): le texte à analyser
        motifs (dict): dictionnaire {nom_motif: pattern_regex}
        nom_famille (str, optionnel): nom du thème global (ex. 'dates_III')
        
    Returns:
        dict: valeurs True/False pour chaque motif et pour la famille
    """
    
    résultats = {}
    
    # Appliquer chaque regex individuellement
    for nom, motif in motifs.items():
        pattern = re.compile(motif, re.I)
        résultats[nom] = bool(pattern.search(texte))
    
    # Valeur agrégée (famille True si au moins un sous-motif est True)
    résultats_global = any(résultats.values())
    
    if nom_famille:
        résultats[f"{nom_famille}_global"] = résultats_global
    
    return résultats


In [ ]:
motifs_evenements_republique = {
    "1870": r"\b1870\b",
    "1875": r"\b1875\b",
    "1905": r"\b1905\b",
    "Front_populaire": r"\b1936\b|\bFront populaire\b",
    "Résistance": r"\b[Rr][ée]sistance\b",
    "1946": r"\b1946\b",
    "1958": r"\b1958\b",
}

In [ ]:
df["analyse_evenements"] = df["texte"].apply(
    lambda t: analyse_regex_famille(t, motifs_evenements_republique, "dates_republique")
)

# Extraire les sous-colonnes
résultats_df = pd.json_normalize(df["analyse_evenements"])
df = pd.concat([df, résultats_df], axis=1)


In [2]:
# Regex des familles des mots pouvant correspondre à des "valeurs de la république"

pattern_valeurs = re.compile(
    r"(libert\w*)"                              # Liberté, libéral, libération...
    r"|(\b[ée]gal\w*)"                          # Égalité, égalitaire...
    r"|(\bfratern\w*)"                          # Fraternité, fraternel...
    r"|(\bindivis\w*)"                          # Indivisible, indivisibilité...
    r"|(\bla[ïi]c\w*)"                          # Laïc, laïcité...
    r"|(\bd[ée]mocrat\w*)"                      # Démocratie, démocratique...
    r"|(\bsoci\w*)"                             # Social, société...
    r"|(\bsouverain\w*)"                        # Souverain, souveraineté...
    r"|(\bunit\w*)"                             # Unité, unification...
    r"|(\bunivers\w*)"                          # Universel, universalisme...
    r"|(\bindiff[ée]r\w*)"                      # Indifférence, indifférent...
    r"|(\bdroit\w*\s*(de\s*l['’]?\s*)?homme\w*)" # Droits de l’homme...
    r"|(\bcitoyen\w*)"                          # Citoyen, citoyenneté...
    r"|(\bddhc\b)"                              # DDHC (Déclaration des Droits de l’Homme et du Citoyen)
    r"|(\bgouvern\w*\s*(du|par|pour)\s+peuple\w*)"  # Gouvernement du peuple...
    r"|(\bchance\w*)"                           # Égalité des chances
    ,
    re.I,
)

def famille_de_mot_valeurs(texte_propre: str) -> bool:
    return bool(pattern_valeurs.search(texte_propre))

In [3]:
# Appliquer sur la colonne
df["texte"] = df["texte"].fillna("") # nécessaire de remplacer les 35 NaN restantes par des chaînes vides pour faire tourner la fonction re
df["FDM_Valeurs"] = df["texte"].apply(famille_de_mot_valeurs)

In [4]:
pattern_figures_sensitive = re.compile(
    r"(\bhussards\s+noirs\b)"
    r"|(\bles\s+lumières\b)"
    r"|(\bVictor\s+Hugo\b)"
    r"|(\bde\s+Gaulle\b)"
    r"|(\bde\s+Gouge\b)"
    r"|(\bAlbert\s+l[’']ouvrier\b)"
    r"|(\bAlexandre\s+Martin\b)"
    r"|(\bEdgar\s+Faure\b)"
    r"|(\bMarianne\b)"
    r"|(\bRobespierre\b)"          
    r"|(\bClémenceau\b)"
    r"|(\bGambetta\b)"              
    r"|(\bVoltaire\b)"
    r"|(\bMontesquieu\b)"
    r"|(\bRousseau\b)"
    r"|(\bJean\s+Moulin\b)"
    r"|(\bles\s+Maquisards\b)"
    r"|(\bZola\b)"
    r"|(\bFerry\b)"
    r"|(\bCondorcet\b)"
    r"|(\bCarnot\b)"
    r"|(\bDelacroix\b)"
    r"|(\bJaur[eè]s\b)"             
    r"|(\bP[eé]guy\b)"              
    r"|(\bBaker\b)"
    r"|(\bRenouvier\b)"
    r"|(\bVeil\b)"
    r"|(\bBlum\b)"
    r"|(\bBriand\b)"
    r"|(\bBrunschvicg\b)"
    r"|(\bCassin\b)"
    r"|(\bCombes\b)"
    r"|(\bDebr[eé]\b)"
    r"|(\bDreyfus\b)"
    r"|(\bHerriot\b)"
    r"|(\bMonnet\b)"
    r"|(\bBadinter\b)"
    r"|(\bLavisse\b)"
    r"|(\bLedru[-\s]?Rollin\b)"     
    r"|(\bMandel\b)"
    r"|(\bMend[eè]s[-\s]?France\b)" 
    r"|(\bProudhon\b)"
    r"|(\bRaspail\b)"
    r"|(\bThiers\b)"
    r"|(\bWaldeck[-\s]?Rousseau\b)"
    r"|(\bJean\s+Zay\b)"
    r"|(\bDumas\b)"
    r"|(\bF[eé]lix\s+[ÉE]bou[eé]\b)" 
    r"|(\bLangevin\b)"
    r"|(\bPainlev[eé]\b)"
    r"|(\bBerthelot\b)"
    r"|(\bMarat\b)"
    r"|(\bPierre\s+Larousse\b)",
    re.I,
)

def famille_de_mot_figures(texte_propre: str) -> bool:
    return bool(pattern_figures_sensitive.search(texte_propre))


In [5]:
# Appliquer sur la colonne
df["texte"] = df["texte"].fillna("") # nécessaire de remplacer les 35 NaN restantes par des chaînes vides pour faire tourner la fonction re
df["FDM_Figures"] = df["texte"].apply(famille_de_mot_figures)

In [6]:
pattern_dates = re.compile(
    r"(\b1789\b)"
    r"|(\b1790\b)"
    r"|(\b1791\b)"
    r"|(\b1792\b)"
    r"|(\b1793\b)"
    r"|(\b1794\b)"
    r"|(\b1795\b)"
    r"|(\b1799\b)"
    r"|(\b1802\b)"
    r"|(\b1804\b)"          
    r"|(\b1815\b)"
    r"|(\b1848\b)"              
    r"|(\b1852\b)"
    r"|(\b1870\b)"
    r"|(\b1875\b)"
    r"|(\b1879\b)"
    r"|(\b1881\b)"
    r"|(\b1882\b)"
    r"|(\b1894\b)"
    r"|(\b1899\b)"
    r"|(\b1901\b)"
    r"|(\b1905\b)"
    r"|(\b1906\b)"
    r"|(\bf[eé]vrier+1934\b)"
    r"|(\b1936\b)"
    r"|(\b1946\b)"
    r"|(\b1954\b)"
    r"|(\b1958\b)"
    r"|(\b1989\b)"
    r"|(\bR[eé]volution\b)"
    r"|(\bR[eé]sistance\b)"
    r"|(\b14+juillet\b)",
    re.I,
)

def dates(texte: str) -> bool:
    """Renvoie True si le texte contient une figure historique républicaine."""
    return bool(pattern_dates.search(texte))

In [7]:
# Appliquer sur la colonne
df["texte"] = df["texte"].fillna("") # nécessaire de remplacer les 35 NaN restantes par des chaînes vides pour faire tourner la fonction re
df["FDM_dates"] = df["texte"].apply(dates)

In [8]:
pattern_dates_historique = re.compile(
    r"|(\bR[ée]volution\b)"
    r"(\b1789\b)"
    r"|(\b1790\b)"
    r"|(\b1791\b)"
    r"|(\b1792\b)"
    r"|(\b1793\b)"
    r"|(\b1794\b)"
    r"|(\b1795\b)"
    r"|(\b1799\b)"
    r"|(\b1802\b)"
    r"|(\b1804\b)"          
    r"|(\b1815\b)"
    r"|(\b1848\b)"              
    r"|(\b1852\b)",
    re.I,
)

def dates_historique(texte: str) -> bool:
    return bool(pattern_dates_historique.search(texte))

In [9]:
# Appliquer sur la colonne
df["texte"] = df["texte"].fillna("") # nécessaire de remplacer les 35 NaN restantes par des chaînes vides pour faire tourner la fonction re
df["FDM_dates_historique"] = df["texte"].apply(dates_historique)

In [10]:
# Regex des dates clés de la IIIe République
pattern_dates_III = re.compile(
    r"(\b1870\b)"
    r"|(\b1875\b)"
    r"|(\b1879\b)"
    r"|(\b1881\b)"
    r"|(\b1882\b)"
    r"|(\b1894\b)"
    r"|(\b1899\b)"
    r"|(\b1901\b)"
    r"|(\b1905\b)"
    r"|(\b1906\b)"
    r"|(\bf[ée]vrier\s*1934\b)"  
    r"|(\b1936\b)",
    re.I,
)

def dates_III(texte: str) -> bool:
    return bool(pattern_dates_III.search(texte))

# Application sur la colonne 'texte'
df["FDM_dates_III"] = df["texte"].apply(dates_III)


In [11]:
pattern_dates_contempo = re.compile(
    r"(\b1946\b)"
    r"|(\b1954\b)"
    r"|(\b1958\b)"
    r"|(\b1989\b)"
    r"|(\bR[ée]sistance\b)",
    re.I,
)

def dates_contempo(texte: str) -> bool:
    """Renvoie True si le texte contient une date ou un repère contemporain."""
    return bool(pattern_dates_contempo.search(texte))

df["FDM_dates_contempo"] = df["texte"].apply(dates_contempo)


In [12]:
df_CL = df[
    ((df["FDM_Valeurs"]) & (df["FDM_Figures"]))
    | ((df["FDM_Valeurs"]) & (df["FDM_dates"]))
    | ((df["FDM_dates"]) & (df["FDM_Figures"]))
]

In [13]:
df_CL

,UID,SeanceRef,SessionRef,dateSeance,dateSeanceJour,numSeanceJour,numSeance,typeAssemblee,legislature,session,...,groupe&gvt_affiliation,groupe_all_affiliation,Texte_clean,repu_match_valide,FDM_Valeurs,FDM_Figures,FDM_dates,FDM_dates_historique,FDM_dates_III,FDM_dates_contempo
144,CRSANR5L15S2018O1N245,NaN,NaN,20180602093000000,samedi 02 juin 2018,1,245,AN,15,Session ordinaire 2017-2018,...,GDR,GDR,"Je maintiens, pour ma part, mon amendement. J'...",False,True,False,True,True,False,True
1111,CRSANR5L15S2018O1N251,NaN,NaN,20180604160000000,lundi 04 juin 2018,1,251,AN,15,Session ordinaire 2017-2018,...,LR,LR,"Il s'agit là d'une mesure de justice, car cet ...",False,True,False,True,True,False,True
1114,CRSANR5L15S2018O1N251,NaN,NaN,20180604160000000,lundi 04 juin 2018,1,251,AN,15,Session ordinaire 2017-2018,...,UDI,UDI,Le 3° de l'article 3 de la loi du 6 juillet 19...,False,True,False,True,True,False,True
1260,CRSANR5L15S2018O1N279,NaN,NaN,20180619150000000,mardi 19 juin 2018,1,279,AN,15,Session ordinaire 2017-2018,...,GVT,NaN,"Monsieur le député Arend, monsieur le présiden...",False,True,True,False,True,False,False
1498,CRSANR5L15S2018O1N279,NaN,NaN,20180619150000000,mardi 19 juin 2018,1,279,AN,15,Session ordinaire 2017-2018,...,LR,LR,"Monsieur le président, madame la garde des sce...",False,True,True,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
683046,CRSANR5L16S2023O1N195,RUANR5L16S2023IDS26941,SCR5A2023O1,20230328213000000,mardi 28 mars 2023,3,195,AN,16,Session ordinaire 2022-2023,...,REN,REN,"…Sébastien Peytavie, Pierre Dharréville, sans ...",False,True,True,False,True,False,False
683124,CRSANR5L16S2024O1N132,RUANR5L16S2024IDS28086,SCR5A2024O1,20240229150000000,jeudi 29 février 2024,2,132,AN,16,Session ordinaire 2023-2024,...,LIOT,LIOT,Je ne suis pas de ceux qui viennent quémander ...,True,True,False,True,True,False,False
683490,CRSANR5L16S2023O1N142,RUANR5L16S2023IDS26788,SCR5A2023O1,20230210150000000,vendredi 10 février 2023,2,142,AN,16,Session ordinaire 2022-2023,...,LFI,LFI,Ce n'est pas beau de mentir ! Je dois dire cep...,False,True,False,True,True,False,True
683553,CRSANR5L16S2023O1N142,RUANR5L16S2023IDS26788,SCR5A2023O1,20230210150000000,vendredi 10 février 2023,2,142,AN,16,Session ordinaire 2022-2023,...,LFI,LFI,"Oui, il y a de la violence, et le fait d'utili...",False,True,False,True,True,False,False


In [14]:
import csv  

df_CL.to_csv(
    "../data/interim/df_CL.csv",
    index=False,
    quoting=csv.QUOTE_ALL,  # a permis de résoudre le soucis d'écart. Checker
)

In [15]:
# Ajoute une colonne FDM_Cooccurrence : True si au moins deux familles sont True
df["FDM_Cooccurrence"] = (
    df[["FDM_Valeurs", "FDM_Figures", "FDM_dates"]].sum(axis=1) >= 2
)


In [30]:
df["CL_République"] = (df["FDM_Cooccurrence"]) | (df["repu_match_valide"])

In [16]:
df["Valeurs"] = (df["FDM_Valeurs"]) & (df["FDM_Cooccurrence"])

In [17]:
df["Dates"] = (df["FDM_dates"]) & (df["FDM_Cooccurrence"])

In [18]:
df["Dates_historique"] = (df["FDM_dates_historique"]) & (df["FDM_Cooccurrence"])

In [19]:
df["Dates_III"] = (df["FDM_dates_III"]) & (df["FDM_Cooccurrence"])

In [20]:
df["Dates_contempo"] = (df["FDM_dates_contempo"]) & (df["FDM_Cooccurrence"])

In [21]:
df["Figures"] = (df["FDM_Figures"]) & (df["FDM_Cooccurrence"])

In [32]:
import csv  

df.to_csv(
    "../data/interim/df_CL.csv",
    index=False,
    quoting=csv.QUOTE_ALL,  # a permis de résoudre le soucis d'écart. Checker
)

In [31]:
df

,UID,SeanceRef,SessionRef,dateSeance,dateSeanceJour,numSeanceJour,numSeance,typeAssemblee,legislature,session,...,FDM_dates_III,FDM_dates_contempo,FDM_Cooccurrence,Valeurs,Dates,Dates_historique,Dates_III,Dates_contempo,Figures,CL_République
0,CRSANR5L15S2018O1N245,NaN,NaN,20180602093000000,samedi 02 juin 2018,1,245,AN,15,Session ordinaire 2017-2018,...,False,False,False,False,False,False,False,False,False,False
1,CRSANR5L15S2018O1N245,NaN,NaN,20180602093000000,samedi 02 juin 2018,1,245,AN,15,Session ordinaire 2017-2018,...,False,False,False,False,False,False,False,False,False,True
2,CRSANR5L15S2018O1N245,NaN,NaN,20180602093000000,samedi 02 juin 2018,1,245,AN,15,Session ordinaire 2017-2018,...,False,False,False,False,False,False,False,False,False,False
3,CRSANR5L15S2018O1N245,NaN,NaN,20180602093000000,samedi 02 juin 2018,1,245,AN,15,Session ordinaire 2017-2018,...,False,False,False,False,False,False,False,False,False,False
4,CRSANR5L15S2018O1N245,NaN,NaN,20180602093000000,samedi 02 juin 2018,1,245,AN,15,Session ordinaire 2017-2018,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
683655,CRSANR5L16S2023O1N156,RUANR5L16S2023IDS26837,SCR5A2023O1,20230227160000000,lundi 27 février 2023,1,156,AN,16,Session ordinaire 2022-2023,...,False,False,False,False,False,False,False,False,False,False
683656,CRSANR5L16S2023O1N156,RUANR5L16S2023IDS26837,SCR5A2023O1,20230227160000000,lundi 27 février 2023,1,156,AN,16,Session ordinaire 2022-2023,...,False,False,False,False,False,False,False,False,False,False
683657,CRSANR5L16S2023O1N156,RUANR5L16S2023IDS26837,SCR5A2023O1,20230227160000000,lundi 27 février 2023,1,156,AN,16,Session ordinaire 2022-2023,...,False,False,False,False,False,False,False,False,False,False
683658,CRSANR5L16S2023O1N156,RUANR5L16S2023IDS26837,SCR5A2023O1,20230227160000000,lundi 27 février 2023,1,156,AN,16,Session ordinaire 2022-2023,...,False,False,False,False,False,False,False,False,False,False


In [24]:
df_valeurs = df[df["FDM_Valeurs"] == True]

In [25]:
df_valeurs

,UID,SeanceRef,SessionRef,dateSeance,dateSeanceJour,numSeanceJour,numSeance,typeAssemblee,legislature,session,...,FDM_dates_historique,FDM_dates_III,FDM_dates_contempo,FDM_Cooccurrence,Valeurs,Dates,Dates_historique,Dates_III,Dates_contempo,Figures
0,CRSANR5L15S2018O1N245,NaN,NaN,20180602093000000,samedi 02 juin 2018,1,245,AN,15,Session ordinaire 2017-2018,...,True,False,False,False,False,False,False,False,False,False
1,CRSANR5L15S2018O1N245,NaN,NaN,20180602093000000,samedi 02 juin 2018,1,245,AN,15,Session ordinaire 2017-2018,...,True,False,False,False,False,False,False,False,False,False
4,CRSANR5L15S2018O1N245,NaN,NaN,20180602093000000,samedi 02 juin 2018,1,245,AN,15,Session ordinaire 2017-2018,...,True,False,False,False,False,False,False,False,False,False
5,CRSANR5L15S2018O1N245,NaN,NaN,20180602093000000,samedi 02 juin 2018,1,245,AN,15,Session ordinaire 2017-2018,...,True,False,False,False,False,False,False,False,False,False
11,CRSANR5L15S2018O1N245,NaN,NaN,20180602093000000,samedi 02 juin 2018,1,245,AN,15,Session ordinaire 2017-2018,...,True,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
683644,CRSANR5L16S2023O1N156,RUANR5L16S2023IDS26837,SCR5A2023O1,20230227160000000,lundi 27 février 2023,1,156,AN,16,Session ordinaire 2022-2023,...,True,False,False,False,False,False,False,False,False,False
683645,CRSANR5L16S2023O1N156,RUANR5L16S2023IDS26837,SCR5A2023O1,20230227160000000,lundi 27 février 2023,1,156,AN,16,Session ordinaire 2022-2023,...,True,False,False,False,False,False,False,False,False,False
683655,CRSANR5L16S2023O1N156,RUANR5L16S2023IDS26837,SCR5A2023O1,20230227160000000,lundi 27 février 2023,1,156,AN,16,Session ordinaire 2022-2023,...,True,False,False,False,False,False,False,False,False,False
683656,CRSANR5L16S2023O1N156,RUANR5L16S2023IDS26837,SCR5A2023O1,20230227160000000,lundi 27 février 2023,1,156,AN,16,Session ordinaire 2022-2023,...,True,False,False,False,False,False,False,False,False,False


In [26]:
df_répu_in = df[df["repu_match_valide"] == True]

In [27]:
df_répu_in

,UID,SeanceRef,SessionRef,dateSeance,dateSeanceJour,numSeanceJour,numSeance,typeAssemblee,legislature,session,...,FDM_dates_historique,FDM_dates_III,FDM_dates_contempo,FDM_Cooccurrence,Valeurs,Dates,Dates_historique,Dates_III,Dates_contempo,Figures
1,CRSANR5L15S2018O1N245,NaN,NaN,20180602093000000,samedi 02 juin 2018,1,245,AN,15,Session ordinaire 2017-2018,...,True,False,False,False,False,False,False,False,False,False
81,CRSANR5L15S2018O1N245,NaN,NaN,20180602093000000,samedi 02 juin 2018,1,245,AN,15,Session ordinaire 2017-2018,...,True,False,False,False,False,False,False,False,False,False
82,CRSANR5L15S2018O1N245,NaN,NaN,20180602093000000,samedi 02 juin 2018,1,245,AN,15,Session ordinaire 2017-2018,...,True,False,False,False,False,False,False,False,False,False
531,CRSANR5L15S2019O1N178,NaN,NaN,20190314150000000,jeudi 14 mars 2019,2,178,AN,15,Session ordinaire 2018-2019,...,True,False,False,False,False,False,False,False,False,False
1314,CRSANR5L15S2018O1N279,NaN,NaN,20180619150000000,mardi 19 juin 2018,1,279,AN,15,Session ordinaire 2017-2018,...,True,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
683309,CRSANR5L16S2024O1N132,RUANR5L16S2024IDS28086,SCR5A2024O1,20240229150000000,jeudi 29 février 2024,2,132,AN,16,Session ordinaire 2023-2024,...,True,False,False,False,False,False,False,False,False,False
683522,CRSANR5L16S2023O1N142,RUANR5L16S2023IDS26788,SCR5A2023O1,20230210150000000,vendredi 10 février 2023,2,142,AN,16,Session ordinaire 2022-2023,...,True,False,False,False,False,False,False,False,False,False
683527,CRSANR5L16S2023O1N142,RUANR5L16S2023IDS26788,SCR5A2023O1,20230210150000000,vendredi 10 février 2023,2,142,AN,16,Session ordinaire 2022-2023,...,True,False,False,False,False,False,False,False,False,False
683539,CRSANR5L16S2023O1N142,RUANR5L16S2023IDS26788,SCR5A2023O1,20230210150000000,vendredi 10 février 2023,2,142,AN,16,Session ordinaire 2022-2023,...,True,False,False,False,False,False,False,False,False,False
